In [30]:
import pandas as pd
import numpy as np
import os
import glob
import statsmodels.api as sm


In [27]:
# path to folder
folder_path = "./Task2Data/*"

# get list of csv files
files = glob.glob(folder_path)

structural = {}
functional = {}

for file in files:
    
    name = os.path.basename(file)
    subject = name.split("_")[0]   # extracts subject ID
    
    mat = pd.read_csv(file, comment='#', header=None).values    
    
    if "WFA" in name:
        structural[subject] = mat
        
    elif "rsfMRI" in name:
        functional[subject] = mat

print(len(structural), len(functional))

19 19


In [28]:
def indirect_connectivity(S):

    n = S.shape[0]
    T = np.zeros_like(S)

    for i in range(n):
        for j in range(n):

            if i == j:
                continue

            vals = []

            for k in range(n):

                if S[i,k] > 0 and S[k,j] > 0:
                    vals.append(min(S[i,k], S[k,j]))

            if vals:
                T[i,j] = max(vals)

    return T

In [29]:
T_matrices = {}

for subject in structural:
    
    S = structural[subject]
    T = indirect_connectivity(S)
    
    T_matrices[subject] = T

T_matrices


{'44': array([[0.        , 0.42372771, 0.42372771, ..., 0.        , 0.        ,
         0.42372771],
        [0.42372771, 0.        , 0.45586847, ..., 0.24253516, 0.41517654,
         0.45400313],
        [0.42372771, 0.45586847, 0.        , ..., 0.24253516, 0.44579148,
         0.48548857],
        ...,
        [0.        , 0.24253516, 0.24253516, ..., 0.        , 0.25487053,
         0.25487053],
        [0.        , 0.41517654, 0.44579148, ..., 0.25487053, 0.        ,
         0.44579148],
        [0.42372771, 0.45400313, 0.48548857, ..., 0.25487053, 0.44579148,
         0.        ]]),
 '37': array([[0.        , 0.44005475, 0.43888443, ..., 0.        , 0.        ,
         0.43210102],
        [0.44005475, 0.        , 0.51677136, ..., 0.16746776, 0.44115001,
         0.44522546],
        [0.43888443, 0.51677136, 0.        , ..., 0.2504874 , 0.44115001,
         0.43489691],
        ...,
        [0.        , 0.16746776, 0.2504874 , ..., 0.        , 0.28143886,
         0.28143886],


In [32]:
# Extract all subjects 
subjects = list(structural.keys())

# Extract edge values across subjects 
f_ij = np.array([functional[s][i,j] for s in subjects])
s_ij = np.array([structural[s][i,j] for s in subjects])
t_ij = np.array([T_matrices[s][i,j] for s in subjects])

Model 1: fij = αij + βij sij

In [42]:
results_model1 = {}

for i in range(68):
    for j in range(68):

        if i == j:
            continue

        # functional connectivity across subjects
        f_ij = np.array([functional[s][i, j] for s in subjects])

        # structural connectivity across subjects
        s_ij = np.array([structural[s][i, j] for s in subjects])

        # skip edges with no structural connectivity
        if np.all(s_ij == 0):
            continue

        # regression design matrix
        X = sm.add_constant(s_ij)

        # fit regression
        model = sm.OLS(f_ij, X).fit()

        # store coefficients (alpha, beta)
        results_model1[(i, j)] = model.params


results_model1 # array containing regression coefficients (intercept, slop) for each key (brain connection)

{(0, 2): array([-0.00671573, -0.12098989]),
 (0, 3): array([-0.010518  ,  0.18157547]),
 (0, 5): array([-0.01684156,  0.04200099]),
 (0, 6): array([-0.11721379,  0.16913724]),
 (0, 7): array([-0.14717242,  0.38025625]),
 (0, 8): array([ 0.098533  , -0.19283825]),
 (0, 9): array([-0.0477041 ,  0.13937877]),
 (0, 11): array([ 3.34794128e-02, -8.41256964e-05]),
 (0, 12): array([-0.00744959,  0.19058379]),
 (0, 13): array([-0.17060806,  0.01672506]),
 (0, 14): array([-1.78411278e-05,  9.74341049e-02]),
 (0, 15): array([-0.03666109,  0.05850299]),
 (0, 16): array([-0.07419885, -0.17645812]),
 (0, 19): array([ 0.00934818, -0.07506927]),
 (0, 20): array([ 0.0169648 , -0.09117565]),
 (0, 21): array([-0.00713586,  0.02948443]),
 (0, 22): array([-0.05104725,  0.10715788]),
 (0, 23): array([-0.27519119,  0.73868014]),
 (0, 26): array([ 0.0348504 , -0.07521936]),
 (0, 27): array([ 0.42016813, -1.01132041]),
 (0, 28): array([-0.188558  ,  0.17716035]),
 (0, 29): array([ 0.04635102, -0.23247751]),
 

Model 2: fij = αij + βij sij + γij sij2

In [43]:
results_model2 = {}

for i in range(68):
    for j in range(68):

        if i == j:
            continue

        # functional connectivity across subjects
        f_ij = np.array([functional[s][i, j] for s in subjects])

        # structural connectivity across subjects
        s_ij = np.array([structural[s][i, j] for s in subjects])

        # skip edges with no structural connectivity
        if np.all(s_ij == 0):
            continue

        # create squared term
        s2_ij = s_ij ** 2

        # regression design matrix
        X = np.column_stack((s_ij, s2_ij))
        X = sm.add_constant(X)

        # fit regression
        model = sm.OLS(f_ij, X).fit()

        # store coefficients (alpha, beta, gamma)
        results_model2[(i, j)] = model.params

results_model2


{(0, 2): array([-0.00647314, -0.3522731 ,  0.57329445]),
 (0, 3): array([-0.010518  ,  0.15803198,  0.06099693]),
 (0, 5): array([-0.02201781,  0.32401903, -0.71849792]),
 (0, 6): array([ 0.25332297, -2.11805154,  3.42016458]),
 (0, 7): array([-1.01716986,  5.76077375, -8.23204022]),
 (0, 8): array([ -2.25550082,  11.28470702, -13.92283309]),
 (0, 9): array([-0.05323632,  0.21945186, -0.17864298]),
 (0, 11): array([ 0.03107781,  0.77044702, -2.01374906]),
 (0, 12): array([-0.00744959,  0.16403343,  0.06599353]),
 (0, 13): array([ 0.88332377, -7.44000376, 12.95003345]),
 (0, 14): array([ 1.15283475e-03, -1.21713879e+00,  3.36771661e+00]),
 (0, 15): array([-0.03333338, -0.34509312,  0.99619443]),
 (0, 16): array([-0.07419885, -0.14809781, -0.06480818]),
 (0, 19): array([ 0.0100623 , -1.05350923,  2.53248803]),
 (0, 20): array([ 0.01322308,  0.24105084, -0.80994921]),
 (0, 21): array([ 1.78221982e-03, -1.15933972e+00,  2.78113684e+00]),
 (0, 22): array([-0.91975598,  4.45757527, -5.430017

Model 3:  fij = αij + βij tij

In [44]:
results_model3 = {}

for i in range(68):
    for j in range(68):

        if i == j:
            continue

        # functional connectivity across subjects
        f_ij = np.array([functional[s][i, j] for s in subjects])

        # indirect structural connectivity across subjects
        t_ij = np.array([T_matrices[s][i, j] for s in subjects])

        # skip edges with no indirect structural connectivity
        if np.all(t_ij == 0):
            continue

        # regression design matrix
        X = sm.add_constant(t_ij)

        # fit regression
        model = sm.OLS(f_ij, X).fit()

        # store coefficients (alpha, beta, gamma)
        results_model3[(i, j)] = model.params

results_model3


{(0, 1): array([-0.17216623,  0.39405373]),
 (0, 2): array([ 0.12027006, -0.34296731]),
 (0, 3): array([-0.0866877 ,  0.22165739]),
 (0, 4): array([ 0.10310384, -0.21099233]),
 (0, 5): array([-0.31150385,  0.70939157]),
 (0, 6): array([-0.47973146,  0.99864639]),
 (0, 7): array([-0.25414271,  0.5459997 ]),
 (0, 8): array([0.00820744, 0.0231279 ]),
 (0, 9): array([-0.04211902,  0.09431778]),
 (0, 10): array([-0.23685531,  0.58715676]),
 (0, 11): array([ 0.39875816, -0.93799679]),
 (0, 12): array([-0.19009386,  0.48148992]),
 (0, 13): array([-0.21142382,  0.10509055]),
 (0, 14): array([-0.10600815,  0.30760846]),
 (0, 15): array([ 0.19992851, -0.52846114]),
 (0, 16): array([ 0.42820184, -1.21958717]),
 (0, 17): array([ 0.22403877, -0.60323534]),
 (0, 18): array([-0.19145356,  0.30161155]),
 (0, 19): array([ 0.09588099, -0.24470373]),
 (0, 20): array([-0.11422434,  0.24048594]),
 (0, 21): array([-0.00181019,  0.00975295]),
 (0, 22): array([ 0.05597897, -0.14546886]),
 (0, 23): array([0.00

Model 4: fij = αij + βij tij + γij tij2

In [ ]:
results_model4 = {}

for i in range(68):
    for j in range(68):

        if i == j:
            continue

        # functional connectivity across subjects
        f_ij = np.array([functional[s][i, j] for s in subjects])

        # indirect structural connectivity across subjects
        t_ij = np.array([T_matrices[s][i, j] for s in subjects])

        # skip edges with no indirect structural connectivity
        if np.all(t_ij == 0):
            continue

        # create squared term
        t2_ij = t_ij ** 2

        # regression design matrix
        X = np.column_stack((t_ij, t2_ij))
        X = sm.add_constant(X)

        # fit regression
        model = sm.OLS(f_ij, X).fit()

        # store coefficients (alpha, beta, gamma)
        results_model4[(i, j)] = model.params

results_model4


Model 5: fij = αij + βij sij + γij tij

In [45]:
results_model5 = {}

for i in range(68):
    for j in range(68):

        if i == j:
            continue

        # functional connectivity across subjects
        f_ij = np.array([functional[s][i, j] for s in subjects])

        # indirect structural connectivity across subjects
        t_ij = np.array([T_matrices[s][i, j] for s in subjects])

        # structural connectivity across subjects
        s_ij = np.array([structural[s][i, j] for s in subjects])

        # skip edges with no direct and indirect structural connectivity
        if np.all(s_ij == 0) and np.all(t_ij == 0):
            continue

        # regression design matrix
        X = np.column_stack((s_ij, t_ij))
        X = sm.add_constant(X)

        # fit regression
        model = sm.OLS(f_ij, X).fit()

        # store coefficients (alpha, beta, gamma)
        results_model5[(i, j)] = model.params

results_model5


{(0, 1): array([-0.17216623,  0.        ,  0.39405373]),
 (0, 2): array([ 0.18822166, -0.12775626, -0.44304566]),
 (0, 3): array([-0.06317687,  0.16307271,  0.14720496]),
 (0, 4): array([ 0.10310384,  0.        , -0.21099233]),
 (0, 5): array([-0.31361084,  0.0275678 ,  0.69469905]),
 (0, 6): array([-0.49289923,  0.06252542,  0.97764129]),
 (0, 7): array([-0.44380563,  0.46935475,  0.61189866]),
 (0, 8): array([ 0.08160507, -0.19967197,  0.04513643]),
 (0, 9): array([-0.00945086,  0.15121808, -0.09881672]),
 (0, 10): array([-0.23685531,  0.        ,  0.58715676]),
 (0, 11): array([ 0.39882817,  0.00595472, -0.94087397]),
 (0, 12): array([-0.1504187 ,  0.15388275,  0.37075443]),
 (0, 13): array([-0.21696663,  0.01831035,  0.10548407]),
 (0, 14): array([-0.09476063,  0.06649612,  0.2657206 ]),
 (0, 15): array([ 0.18684464,  0.07354754, -0.55588954]),
 (0, 16): array([ 0.40296573, -0.12760888, -1.15174025]),
 (0, 17): array([ 0.22403877,  0.        , -0.60323534]),
 (0, 18): array([-0.191